<a href="https://colab.research.google.com/github/nasif-raihan/GSU/blob/main/CSCI_7090_Module_9_Homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 9 — Homework
**Computer Vision · Object Detection Basics**

Your boxes, grid size, thresholds, and candidate detections are generated from your **roll number**, so your numbers are unique — pasted generic answers will not match your outputs.

## Rules
- Set `ROLL_NUMBER` at the top, then run the setup cell. Do **not** edit the parameter lines.
- Each question is a **task**: write the code yourself and store the result in the named variable(s) under **Produce**. There are no fill-in-the-blank stubs.
- Answer the **Written** prompts (Q7) in your own words. Run the final **Submission Summary** and keep its output.

## Grading (100): Q1 15 · Q2 15 · Q3 10 · Q4 20 · Q5 10 · Q6 15 · Q7 15  ·  ~1.5 hours

## Setup — run first
Generates your boxes, thresholds, and a set of candidate detections, plus an `iou(a, b)` helper (boxes are in corner format `(x_min, y_min, x_max, y_max)`).

In [1]:
# === SETUP — run first ===
ROLL_NUMBER = "901503252"
import hashlib, numpy as np
seed = int(hashlib.sha256(ROLL_NUMBER.encode()).hexdigest(), 16) % (2**32)
rng  = np.random.default_rng(seed)

# ---- Your personal parameters (unique to your roll number) — DO NOT EDIT ----
BOX  = np.array([rng.integers(0,60), rng.integers(0,60), rng.integers(120,200), rng.integers(120,200)])  # ground-truth box (corner fmt)
PRED = BOX + rng.integers(-25, 26, size=4)                    # a predicted box near it
GRID       = int(rng.choice([3, 4, 5]))
N_SCALES   = int(rng.choice([2, 3]))
N_RATIOS   = int(rng.choice([2, 3]))
IOU_THRESH = round(float(rng.choice([0.3, 0.5, 0.7])), 2)
SCORE_THRESH = round(float(rng.choice([0.4, 0.5, 0.6])), 2)    # for the Q6 pipeline

# A set of candidate detections for Q6 — a few clusters of overlapping boxes (so NMS has duplicates to remove)
rng9 = np.random.default_rng(seed + 9)
_cl = int(rng9.integers(3, 5))               # number of object clusters
_bx, _sc = [], []
for _ in range(_cl):
    bx, by = int(rng9.integers(0, 120)), int(rng9.integers(0, 120))
    bw, bh = int(rng9.integers(45, 75)), int(rng9.integers(45, 75))
    for _ in range(int(rng9.integers(2, 4))):   # overlapping near-duplicates
        jx, jy = int(rng9.integers(-6, 7)), int(rng9.integers(-6, 7))
        _bx.append([bx+jx, by+jy, bx+bw+jx, by+bh+jy]); _sc.append(round(float(rng9.uniform(0.2, 0.99)), 2))
CAND_BOXES  = np.array(_bx)
CAND_SCORES = np.array(_sc)
M = len(_bx)

def iou(a, b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    ua=(a[2]-a[0])*(a[3]-a[1]); ub=(b[2]-b[0])*(b[3]-b[1])
    return inter/(ua+ub-inter) if (ua+ub-inter)>0 else 0.0

print("Roll:", ROLL_NUMBER, "| seed:", seed)
print("BOX:", BOX, "| PRED:", PRED)
print(f"GRID={GRID}  N_SCALES={N_SCALES}  N_RATIOS={N_RATIOS}  IOU_THRESH={IOU_THRESH}  SCORE_THRESH={SCORE_THRESH}")
print(f"Q6 candidates: {M} boxes | scores {CAND_SCORES}")

Roll: 901503252 | seed: 279384723
BOX: [  2  51 138 162] | PRED: [ -6  29 147 162]
GRID=5  N_SCALES=3  N_RATIOS=2  IOU_THRESH=0.5  SCORE_THRESH=0.4
Q6 candidates: 8 boxes | scores [0.35 0.98 0.71 0.86 0.57 0.49 0.39 0.73]


## Q1 — Coordinate formats (15 pts)
Convert **your** `BOX` from corner format `(x_min, y_min, x_max, y_max)` to center format `(cx, cy, w, h)`.

**Produce:** `center` (a length-4 array `[cx, cy, w, h]`)

In [2]:
# Convert BOX (corner) -> center format. Produce: center
# Corner format: (x_min, y_min, x_max, y_max)
# Center format: (cx, cy, w, h)
x_min, y_min, x_max, y_max = BOX
w = x_max - x_min
h = y_max - y_min
cx = x_min + w / 2.0
cy = y_min + h / 2.0
center = np.array([cx, cy, w, h])
print("center:", center)

center: [ 70.  106.5 136.  111. ]


## Q2 — IoU (15 pts)
Compute the **intersection area** between `BOX` and `PRED` yourself (careful when the boxes don't overlap), then the IoU using the provided `iou` helper.

**Produce:** `inter_area` and `iou_val`

In [3]:
# Compute inter_area (clamp at 0 when disjoint) and iou_val = iou(BOX, PRED).
# Produce: inter_area, iou_val

x1 = max(BOX[0], PRED[0])
y1 = max(BOX[1], PRED[1])
x2 = min(BOX[2], PRED[2])
y2 = min(BOX[3], PRED[3])

inter_area = max(0, x2 - x1) * max(0, y2 - y1)
iou_val = iou(BOX, PRED)

print("inter_area:", inter_area, "iou_val:", iou_val)

inter_area: 15096 iou_val: 0.7418546365914787


## Q3 — Counting anchors (10 pts)
If anchors are placed at every cell of a `GRID×GRID` feature map, with `N_SCALES` scales and `N_RATIOS` aspect ratios per cell, how many anchors are there in total?

**Produce:** `total_anchors`

In [4]:
# Total anchors over the GRIDxGRID map. Produce: total_anchors
total_anchors = GRID * GRID * N_SCALES * N_RATIOS
print("total_anchors:", total_anchors)

total_anchors: 150


## Q4 — Non-Maximum Suppression (20 pts)
Write `nms(boxes, scores, thr)`: repeatedly take the highest-scoring box, keep it, and drop every remaining box whose IoU with it is **≥ `thr`**. Return the kept indices. Run it on the demo set below with **your** `IOU_THRESH`.

**Produce:** the `nms` function and `kept`

In [5]:
demo_boxes  = np.array([[50,40,180,200],[60,50,190,210],[55,45,185,205],[300,300,360,360]])
demo_scores = np.array([0.90, 0.80, 0.75, 0.95])
# Write nms(boxes, scores, thr) and apply it. Produce: nms, kept

def nms(boxes, scores, thr):
    if len(boxes) == 0:
        return []

    # Get indices of sorted scores in descending order
    order = np.argsort(scores)[::-1]
    keep = []

    while len(order) > 0:
        best_idx = order[0]
        keep.append(best_idx)

        if len(order) == 1:
            break

        # Compute IoUs of the remaining boxes with the best box
        best_box = boxes[best_idx]
        rem_boxes = boxes[order[1:]]

        ious = np.array([iou(best_box, b) for b in rem_boxes])

        # Keep indices where IoU is strictly less than threshold
        below_thr_indices = np.where(ious < thr)[0]

        # Update order to match those remaining indexes
        order = order[below_thr_indices + 1]

    return keep

kept = nms(demo_boxes, demo_scores, IOU_THRESH)
print("kept:", kept)

kept: [np.int64(3), np.int64(0)]


## Q5 — Anchor matching (10 pts)
An anchor is a **positive** match to the ground-truth `BOX` if its IoU exceeds `IOU_THRESH`. Decide whether `PRED` would be matched.

**Produce:** `matched` (a bool)

In [6]:
# matched = is iou(BOX, PRED) above IOU_THRESH? Produce: matched
matched = bool(iou_val > IOU_THRESH)
print("matched:", matched)

matched: True


## Q6 — Mini detection pipeline (15 pts)
Put the pieces together on **your** candidate detections (`CAND_BOXES`, `CAND_SCORES`): first **filter out** boxes whose score is below `SCORE_THRESH`, then run your `nms` on what remains using `IOU_THRESH`. Report how many detections survive each stage.

**Produce:** `n_after_score` (count after the score filter) and `n_final` (count after NMS)

In [7]:
# 1) keep boxes with score >= SCORE_THRESH  2) run nms on them with IOU_THRESH.
# Produce: n_after_score, n_final

# Filter out indices
score_filter_indices = np.where(CAND_SCORES >= SCORE_THRESH)[0]
n_after_score = len(score_filter_indices)

filtered_boxes = CAND_BOXES[score_filter_indices]
filtered_scores = CAND_SCORES[score_filter_indices]

# Run NMS
final_kept_indices = nms(filtered_boxes, filtered_scores, IOU_THRESH)
n_final = len(final_kept_indices)

print("n_after_score:", n_after_score, "n_final:", n_final)

n_after_score: 6 n_final: 3


## Q7 — Short answer, no code (15 pts)
Answer in your own words:

**(a)** Why do detectors usually predict box **offsets** to a reference (anchor) box rather than absolute coordinates?

> Predicting relative offsets stabilizes and simplifies network learning. It restricts the bounding box search space and allows the model to leverage translation-invariant convolutional representations of anchor boundaries.

**(b)** What does **IoU = 1** mean, and what does **IoU = 0** mean?

> **IoU = 1** indicates perfect overlap; the prediction and ground-truth boxes are identical in both location and shape. **IoU = 0** means there is zero overlap between the two boxes.

**(c)** What problem does **Non-Maximum Suppression** solve, and what happens if the IoU threshold is set **too high** (near 1.0)?

> NMS eliminates redundant, overlapping bounding boxes representing the same detected object, keeping only the highest-scoring candidate. If the IoU threshold is set too high (near 1.0), almost no overlapping boxes will be filtered, resulting in many redundant duplicate detections for the same object.

**(d)** SSD places anchors on **several feature maps of different resolutions**. Why use multiple scales instead of just one?

> Using multiple scales allows SSD to detect objects of varying sizes. Shallow (high-resolution) feature maps are responsible for detecting smaller objects, while deep (low-resolution) feature maps detect larger objects.

## Submission Summary
Run last; keep the output.

In [8]:
# === SUBMISSION SUMMARY ===
print("Roll:", ROLL_NUMBER, "| seed:", seed)
print("Q1 center:", center)
print("Q2 inter_area / IoU:", inter_area, iou_val)
print("Q3 total_anchors:", total_anchors, "| (grid,scales,ratios)", (GRID,N_SCALES,N_RATIOS))
print("Q4 NMS kept:", kept, "| IOU_THRESH", IOU_THRESH)
print("Q5 matched:", matched)
print("Q6 pipeline:", M, "->", n_after_score, "->", n_final, "| SCORE_THRESH", SCORE_THRESH)

Roll: 901503252 | seed: 279384723
Q1 center: [ 70.  106.5 136.  111. ]
Q2 inter_area / IoU: 15096 0.7418546365914787
Q3 total_anchors: 150 | (grid,scales,ratios) (5, 3, 2)
Q4 NMS kept: [np.int64(3), np.int64(0)] | IOU_THRESH 0.5
Q5 matched: True
Q6 pipeline: 8 -> 6 -> 3 | SCORE_THRESH 0.4
